# RNA sequence and structural-motif coverage

This notebook produces **one four-panel figure**: coverage curves, two UMAP maps, and structural descriptor shifts. Run all cells in order in a fresh kernel.

**Setup:** Python 3.12; install `requirements.txt`, then launch `jupyter lab` in this directory. The supplied `mdRNAcat_pdb_ids.txt` is the sole authority for membership (48 PDB IDs). Bundled input tables allow the default run to work offline. Set `REFRESH_INPUTS = True` to download public RCSB and RNA 3D Hub data instead; no credentials are required. A refresh may take tens of minutes.

**Scope:** X-ray background, 40–192 nt, resolution ≤3.6 Å, initial release by 2026-09-25. Representatives bypass background eligibility filters. These are deposited sequences/annotations, not reconstructed simulation inputs. Selection membership is never inferred from other entries sharing a sequence.

**Reproducibility:** cache keys include the query and input IDs. Only explicitly targeted annotation files are processed. Fixed UMAP settings replace exploratory clustering/tuning. A date filter limits initial release dates; it does not restore historical versions of records. Bundled inputs have verified SHA-256 checksums. Archive refreshed `cache/` and `results/` alongside a manuscript version for exact input provenance. Live downloads and the removal of stale-file dependence can change values from earlier figures.

**Structural interpretation:** this version preserves the source notebook's loop-annotated-residue denominator and entity aggregation (counts summed across chains; lengths/fractions maximized; length moments averaged). It does not resolve possible chain-multiplicity bias. Chain identifiers are mapped explicitly through RCSB author-chain identifiers, preserving case. Missing representative annotations or ambiguous mappings stop execution rather than silently shrinking the cohort. Nonfinite distance ratios are counted and excluded explicitly.

In [ ]:
import csv
import hashlib
import io
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import ultraplot as uplt
import umap
from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

CONFIG = dict(release_date_max="2026-09-25", min_length=40, max_length=192,
              max_resolution=3.6, background_method="X-RAY DIFFRACTION",
              poly_u_fraction=0.65, poly_u_run=16, schema_version=1)
SEED = 42
WORKERS = 6
OFFLINE = False  # Used only for optional public-data refresh.
REFRESH_INPUTS = False  # Default: recompute the figure from the bundled input tables.
DATA = Path("data")
ID_FILE = Path("mdRNAcat_pdb_ids.txt")
selected_ids = {s.strip().upper() for s in ID_FILE.read_text().splitlines() if s.strip()}
if len(selected_ids) != 48 or not all(re.fullmatch(r"[0-9A-Z]{4}", p) for p in selected_ids):
    raise ValueError("Expected exactly 48 distinct four-character PDB IDs.")
run_key = hashlib.sha256(json.dumps([CONFIG, sorted(selected_ids)], sort_keys=True).encode()).hexdigest()[:16]
CACHE = Path("cache") / run_key
RESULTS = Path("results") / run_key
CACHE.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
THRESHOLDS = np.unique(np.r_[np.round(np.linspace(0.5, 3.0, 31), 3), 1.2, 1.5, 2.0])
FEATURES = ["f_hairpin", "f_internal", "f_junction",
            "f_hairpin_res", "f_internal_res", "f_junction_res",
            "mean_len_hairpin", "std_len_hairpin", "mean_len_internal",
            "std_len_internal", "mean_len_junction", "std_len_junction",
            "junction_order_max"]
print("Configured representatives:", len(selected_ids))

if not REFRESH_INPUTS:
    input_manifest = json.loads((DATA / "input_manifest.json").read_text())
    if input_manifest["config"] != CONFIG or input_manifest["selected_pdb_ids"] != sorted(selected_ids):
        raise ValueError("Bundled input configuration differs; restore settings or enable REFRESH_INPUTS.")
    for name, expected in input_manifest["files"].items():
        actual = hashlib.sha256((DATA / name).read_bytes()).hexdigest()
        if actual != expected:
            raise ValueError(f"Bundled input checksum mismatch: {name}")


## 1. Public data and explicit reference population

HTTP failures are retried and never cached as missing annotations. The reference search filters experimental method directly. Successful raw responses are retained by request hash. All RNA entities from selected IDs are fetched independently of reference filters.

In [ ]:
def request_text(url, payload=None):
    key = hashlib.sha256(json.dumps([url, payload], sort_keys=True).encode()).hexdigest()
    path = CACHE / (key + ".txt")
    if path.exists():
        return path.read_text(encoding="utf-8")
    if OFFLINE:
        raise RuntimeError(f"Offline cache miss for {url}")
    for attempt in range(5):
        try:
            response = requests.get(url, timeout=90) if payload is None else requests.post(url, json=payload, timeout=90)
            response.raise_for_status()
            text = response.text
            path.write_text(text, encoding="utf-8")
            return text
        except requests.RequestException:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)


def get_json(url):
    return json.loads(request_text(url))


def terminal(attribute, operator, value):
    return dict(type="terminal", service="text", parameters=dict(attribute=attribute, operator=operator, value=value))


def search_entities(nodes):
    ids = []
    for start in range(0, 200000, 10000):
        payload = dict(query=dict(type="group", logical_operator="and", nodes=nodes),
                       return_type="polymer_entity",
                       request_options=dict(results_content_type=["experimental"],
                                            paginate=dict(start=start, rows=10000)))
        raw = request_text("https://search.rcsb.org/rcsbsearch/v2/query", payload)
        data = json.loads(raw) if raw.strip() else {}
        batch = [r["identifier"] for r in data.get("result_set", [])]
        ids.extend(batch)
        if len(batch) < 10000:
            return sorted(set(ids))
    raise RuntimeError("Search pagination limit reached; increase it explicitly.")


def clean_sequence(raw):
    return re.sub(r"[^A-Z]", "", raw.upper()).replace("T", "U")


def fetch_entity(identifier):
    pdb, entity = identifier.split("_", 1)
    data = get_json(f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb}/{entity}")
    poly = data["entity_poly"]
    sequence = clean_sequence(poly.get("pdbx_seq_one_letter_code_can") or poly.get("pdbx_seq_one_letter_code", ""))
    ids = data["rcsb_polymer_entity_container_identifiers"]
    return dict(pdb_id=pdb.upper(), entity_id=str(entity), sequence=sequence,
                length=len(sequence), auth_asym_ids=ids.get("auth_asym_ids", []))


def background_sequence_ok(sequence):
    if not sequence or set(sequence) - set("ACGU"):
        return False
    if not CONFIG["min_length"] <= len(sequence) <= CONFIG["max_length"]:
        return False
    return not (sequence.count("U") / len(sequence) >= CONFIG["poly_u_fraction"]
                and "U" * CONFIG["poly_u_run"] in sequence)


if REFRESH_INPUTS:
    rna_node = terminal("entity_poly.rcsb_entity_polymer_type", "exact_match", "RNA")
    background_poly_ids = search_entities([
        rna_node,
        terminal("entity_poly.rcsb_sample_sequence_length", "greater_or_equal", CONFIG["min_length"]),
        terminal("entity_poly.rcsb_sample_sequence_length", "less_or_equal", CONFIG["max_length"]),
        terminal("rcsb_entry_info.resolution_combined", "less_or_equal", CONFIG["max_resolution"]),
        terminal("exptl.method", "exact_match", CONFIG["background_method"]),
        terminal("rcsb_accession_info.initial_release_date", "less_or_equal", CONFIG["release_date_max"] + "T23:59:59Z"),
    ])
    selected_poly_ids = search_entities([rna_node, terminal("rcsb_entry_container_identifiers.entry_id", "in", sorted(selected_ids))])
    all_poly_ids = sorted(set(background_poly_ids) | set(selected_poly_ids))
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        records = list(pool.map(fetch_entity, all_poly_ids))
    entity_df = pd.DataFrame(records)
    selected_rows = entity_df.pdb_id.isin(selected_ids)
    eligible = entity_df.sequence.map(background_sequence_ok)
    entity_df = entity_df[selected_rows | eligible].copy().sort_values(["pdb_id", "entity_id"]).reset_index(drop=True)
else:
    entity_df = pd.read_json(DATA / "entities.json", dtype={"pdb_id": str, "entity_id": str})
if (entity_df.sequence == "").any():
    raise ValueError("A selected entity has an empty sequence.")
missing = selected_ids - set(entity_df.pdb_id)
if missing:
    raise ValueError(f"Missing selected entries: {sorted(missing)}")
seq_df = (entity_df.assign(is_selected=entity_df.pdb_id.isin(selected_ids))
          .groupby("sequence", as_index=False)
          .agg(pdb_ids=("pdb_id", lambda x: ",".join(sorted(set(x)))),
               is_selected=("is_selected", "max")))
print(f"Sequence rows: {len(seq_df)}; selected: {seq_df.is_selected.sum()}; evaluation: {(~seq_df.is_selected).sum()}")


## 2. RNA 3D Hub loop descriptors

For each distinct sequence, use the alphabetically first associated entry (prefer a selected entry for selected sequences), and additionally include every selected PDB ID. No other cached entry is read. RNA 3D Hub unit identifiers use author-chain IDs; these are mapped to entity IDs using RCSB entity metadata. Only eligible background entities and RNA entities from the explicit representative list are retained.

Loops contain closing residues as defined by the annotation source; the fractions are not fractions of all polymer residues. Entries with no annotated loops have no descriptor row. The table below reports annotation attrition.

In [ ]:
def parse_loops(text):
    if not text.strip() or text.strip() == "No loops found":
        return []
    if "<html" in text.lower() or "<!doctype" in text.lower():
        raise ValueError("RNA 3D Hub returned HTML instead of loop CSV.")
    loops = []
    for row in csv.reader(io.StringIO(text)):
        if not row:
            continue
        if len(row) < 2 or not re.match(r"^(HL|IL|J[0-9]*)_", row[0]):
            raise ValueError("Unexpected RNA 3D Hub CSV schema.")
        nts = []
        for unit in row[1].split(","):
            fields = unit.strip().split("|")
            if len(fields) < 5:
                raise ValueError(f"Malformed nucleotide identifier: {unit}")
            suffix = next((v for v in reversed(fields[5:]) if v), "")
            nts.append((fields[2], (fields[4], suffix)))
        loops.append((row[0].split("_", 1)[0], nts))
    return loops


def chain_descriptors(pdb, loops):
    chains = {}
    for kind, nts in loops:
        per_chain = {}
        for chain, key in nts:
            per_chain.setdefault(chain, set()).add(key)
        for chain, keys in per_chain.items():
            state = chains.setdefault(chain, dict(all=set(), hairpin=set(), internal=set(), junction=set(),
                                                  lengths={k: [] for k in ["hairpin", "internal", "junction"]}, orders=[]))
            group = "hairpin" if kind == "HL" else "internal" if kind == "IL" else "junction"
            state["all"].update(keys)
            state[group].update(keys)
            state["lengths"][group].append(len(keys))
            if group == "junction" and kind[1:].isdigit():
                state["orders"].append(int(kind[1:]))
    rows = []
    for chain, state in sorted(chains.items()):
        length = len(state["all"])
        row = dict(pdb_id=pdb, chain_id=chain, length=length,
                   junction_order_max=max(state["orders"]) if state["orders"] else np.nan)
        for group in ["hairpin", "internal", "junction"]:
            values = state["lengths"][group]
            row[f"n_{group}"] = len(values)
            row[f"f_{group}_res"] = len(state[group]) / length
            row[f"mean_len_{group}"] = np.mean(values) if values else np.nan
            row[f"std_len_{group}"] = np.std(values) if values else np.nan
        rows.append(row)
    return rows


def fetch_motifs(pdb):
    text = request_text(f"https://rna.bgsu.edu/rna3dhub/loops/download/{pdb}")
    return chain_descriptors(pdb, parse_loops(text))


target_ids = set(selected_ids)
for row in seq_df.itertuples():
    ids = row.pdb_ids.split(",")
    candidates = sorted(set(ids) & selected_ids) if row.is_selected else ids
    target_ids.add(candidates[0])
if REFRESH_INPUTS:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        chain_records = [row for rows in pool.map(fetch_motifs, sorted(target_ids)) for row in rows]
    chain_df = pd.DataFrame(chain_records)
    if chain_df.empty:
        raise ValueError("No loop annotations were returned.")
    chain_map = {}
    for row in entity_df.itertuples():
        for chain in row.auth_asym_ids:
            key = (row.pdb_id, chain)
            if key in chain_map and chain_map[key] != row.entity_id:
                raise ValueError(f"Ambiguous author-chain mapping: {key}")
            chain_map[key] = row.entity_id
    chain_df["entity_id"] = [chain_map.get((p, c)) for p, c in zip(chain_df.pdb_id, chain_df.chain_id)]
    unmapped = chain_df[chain_df.entity_id.isna()]
    unmapped.to_csv(RESULTS / "excluded_unmapped_chains.csv", index=False)
    chain_df = chain_df.dropna(subset=["entity_id"]).copy()
else:
    chain_df = pd.read_csv(DATA / "chain_descriptors.csv", dtype={"pdb_id": str, "entity_id": str, "chain_id": str})
    unmapped = pd.read_csv(DATA / "excluded_unmapped_chains.csv", dtype={"pdb_id": str, "chain_id": str})
aggregation = {"length": "max", "junction_order_max": "max", "chain_id": "nunique"}
for group in ["hairpin", "internal", "junction"]:
    aggregation.update({f"n_{group}": "sum", f"f_{group}_res": "max",
                        f"mean_len_{group}": "mean", f"std_len_{group}": "mean"})
motif_df = chain_df.groupby(["pdb_id", "entity_id"], as_index=False).agg(aggregation).rename(columns={"chain_id": "n_chains"})
for group in ["hairpin", "internal", "junction"]:
    motif_df[f"f_{group}"] = motif_df[f"n_{group}"] / motif_df.length
motif_df["is_selected"] = motif_df.pdb_id.isin(selected_ids)
missing = selected_ids - set(motif_df.pdb_id)
if missing:
    raise ValueError(f"Selected entries without mapped loop descriptors: {sorted(missing)}. Inspect public annotations and excluded_unmapped_chains.csv.")
print(f"Structural rows: {len(motif_df)}; selected rows: {motif_df.is_selected.sum()}; selected PDB IDs: {motif_df.loc[motif_df.is_selected, 'pdb_id'].nunique()}")
print(f"Target entries: {len(target_ids)}; entries with retained descriptors: {motif_df.pdb_id.nunique()}; unmapped/ineligible chains: {len(unmapped)}")


## 3. Coverage in feature space

Sequence distances use SVD-reduced, L2-normalized 3-mer TF–IDF with Euclidean distance. Structural distances use median-imputed, z-scored loop descriptors. Both feature transformations are fitted to the combined population. The local scale is the median distance to ten non-self neighbours in that population, including representatives. Coverage evaluates only non-selected rows with finite ratios. Pointwise intervals resample fixed ratios 300 times; they do not quantify annotation or selection uncertainty. The fixed report thresholds are evaluated directly, not interpolated.

In [ ]:
def coverage_analysis(X, selected, seed):
    selected = np.asarray(selected, dtype=bool)
    if len(X) < 11 or not selected.any() or selected.all():
        raise ValueError("Need at least 11 rows and both representative and evaluation rows.")
    nn = NearestNeighbors(n_neighbors=11, metric="euclidean").fit(X)
    distances, indices = nn.kneighbors(X)
    # Remove self by index, including when tied zero distances reorder neighbours.
    local = np.array([np.median(d[j != i][:10]) for i, (d, j) in enumerate(zip(distances, indices))])
    nearest = pairwise_distances(X, X[selected], metric="euclidean").min(axis=1)
    ratio = np.divide(nearest, local, out=np.full(len(X), np.nan), where=local > 0)
    valid = (~selected) & np.isfinite(ratio)
    values = ratio[valid]
    if not len(values):
        raise ValueError("No finite evaluation ratios.")
    rng = np.random.default_rng(seed)
    bootstrap = values[rng.integers(0, len(values), size=(300, len(values)))]
    curve = []
    for t in THRESHOLDS:
        draws = (bootstrap <= t).mean(axis=1)
        curve.append((t, (values <= t).mean(), *np.quantile(draws, [0.025, 0.975])))
    summary = dict(n_total=len(X), n_selected=int(selected.sum()),
                   n_evaluation=int((~selected).sum()), n_evaluable=int(valid.sum()),
                   n_excluded_nonfinite=int(((~selected) & ~np.isfinite(ratio)).sum()))
    return ratio, pd.DataFrame(curve, columns=["threshold", "coverage", "ci_low", "ci_high"]), summary


vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 3), lowercase=False)
X_tfidf = vectorizer.fit_transform(seq_df.sequence)
n_components = min(80, max(8, X_tfidf.shape[1] - 1))
X_seq = TruncatedSVD(n_components=n_components, random_state=SEED).fit_transform(X_tfidf)
features = [f for f in FEATURES if motif_df[f].notna().mean() >= 0.15 and motif_df[f].std(ddof=0) > 1e-10]
if not features:
    raise ValueError("No supported, nonconstant structural descriptors.")
raw = motif_df[features]
X_struct = StandardScaler().fit_transform(raw.fillna(raw.median()))
seq_ratio, seq_curve, seq_summary = coverage_analysis(X_seq, seq_df.is_selected, SEED)
struct_ratio, struct_curve, struct_summary = coverage_analysis(X_struct, motif_df.is_selected, 77)
selected_struct = motif_df.is_selected.to_numpy()
bg = raw.loc[~selected_struct]
sel = raw.loc[selected_struct]
shifts = ((sel.mean() - bg.mean()) / bg.std(ddof=0).replace(0, np.nan)).dropna()
shifts = shifts.loc[shifts.abs().sort_values(ascending=False).index]
summary = pd.DataFrame([seq_summary, struct_summary], index=["sequence", "structural_motif"])
display(summary)
print("Structural features:", ", ".join(features))
seq_df.assign(ratio=seq_ratio).to_csv(RESULTS / "sequence_ratios.csv", index=False)
motif_df.assign(ratio=struct_ratio).to_csv(RESULTS / "structural_ratios.csv", index=False)
chain_df.to_csv(RESULTS / "chain_descriptors.csv", index=False)
entity_df.to_json(RESULTS / "entities.json", orient="records", indent=2)
seq_curve.to_csv(RESULTS / "sequence_coverage.csv", index=False)
struct_curve.to_csv(RESULTS / "structural_coverage.csv", index=False)
shifts.rename("delta_z").to_csv(RESULTS / "descriptor_shifts.csv")
summary.to_csv(RESULTS / "evaluation_counts.csv")


## 4. The publication coverage figure

UMAP coordinates affect visualization only. Fixed parameters (sequence: 35 neighbours, minimum distance 0.1; structure: 48 neighbours, minimum distance 0.35) and seed 42 replace tuning. Smoothed surfaces use independent scales; structural values are clipped at the 4th and 92nd percentiles for display only. UltraPlot manages the complete layout, colour bars and legend.

In [ ]:
def winsorize_finite(zvals, q_low=0.03, q_high=0.97):
    z = np.asarray(zvals, dtype=float).copy()
    finite = np.isfinite(z)
    if finite.sum() < 20:
        return z
    lo = float(np.quantile(z[finite], q_low))
    hi = float(np.quantile(z[finite], q_high))
    if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
        z[finite] = np.clip(z[finite], lo, hi)
    return z


def build_surface(
    xv,
    yv,
    zv,
    grid_n,
    surf_k,
    mask_quantile,
    support_radius_factor=1.4,
    clip_quantiles=None,
):
    mask_finite = np.isfinite(zv)
    xy = np.column_stack([xv[mask_finite], yv[mask_finite]])
    zvals = zv[mask_finite]
    if xy.shape[0] < 8:
        raise RuntimeError("Too few finite points to build contour surface.")

    if clip_quantiles is not None:
        zvals = winsorize_finite(zvals, q_low=float(clip_quantiles[0]), q_high=float(clip_quantiles[1]))

    xg = np.linspace(xy[:, 0].min(), xy[:, 0].max(), grid_n)
    yg = np.linspace(xy[:, 1].min(), xy[:, 1].max(), grid_n)
    XX, YY = np.meshgrid(xg, yg)
    grid_pts = np.column_stack([XX.ravel(), YY.ravel()])

    k_surf = int(max(3, surf_k))
    nbr = NearestNeighbors(n_neighbors=min(k_surf, len(xy)), metric="euclidean")
    nbr.fit(xy)
    dist_k, idx_k = nbr.kneighbors(grid_pts)

    sigma = np.nanmedian(dist_k[:, -1])
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    weights = np.exp(-(dist_k ** 2) / (2.0 * sigma ** 2)) + 1e-12
    ZZ_flat = (weights * zvals[idx_k]).sum(axis=1) / weights.sum(axis=1)

    density_scale = dist_k[:, -1]
    cutoff = np.quantile(density_scale, mask_quantile)
    ZZ_flat[density_scale > cutoff] = np.nan

    support_nn = NearestNeighbors(n_neighbors=min(8, len(xy)), metric="euclidean")
    support_nn.fit(xy)
    support_dist, _ = support_nn.kneighbors(xy)
    if support_dist.shape[1] > 1:
        support_radius = np.quantile(support_dist[:, 1], 0.90)
    else:
        support_radius = np.quantile(support_dist[:, 0], 0.90)
    ZZ_flat[dist_k[:, 0] > (float(support_radius_factor) * support_radius)] = np.nan

    ZZ = ZZ_flat.reshape(XX.shape)
    return XX, YY, ZZ


def structural_levels_from_data(zvals):
    finite = np.asarray(zvals, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size < 20:
        return [0.6, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]

    qs = np.quantile(finite, [0.05, 0.15, 0.30, 0.48, 0.66, 0.82, 0.93])
    levels = np.unique(np.round(qs, 3))

    if levels.size < 5:
        lo = float(np.nanmin(finite))
        hi = float(np.nanmax(finite))
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            return [0.6, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
        levels = np.linspace(lo, hi, 7)

    return levels.tolist()


def plot_surface_panel(
    ax,
    x,
    y,
    z,
    selected_mask,
    title,
    cmap_name,
    levels,
    contour_levels,
    contour_colors,
    contour_text,
    show_legend=False,
    legend_loc="upper left",
    surf_k_override=None,
    mask_quantile_override=None,
    support_radius_factor=1.4,
    clip_quantiles=None,
):
    grid_n = 180
    surf_k = 24 if surf_k_override is None else int(surf_k_override)
    mask_quantile = 0.82 if mask_quantile_override is None else float(mask_quantile_override)

    XX, YY, ZZ = build_surface(
        x,
        y,
        z,
        grid_n=grid_n,
        surf_k=surf_k,
        mask_quantile=mask_quantile,
        support_radius_factor=support_radius_factor,
        clip_quantiles=clip_quantiles,
    )

    cf = ax.contourf(XX, YY, ZZ, levels=levels, cmap=cmap_name, alpha=0.72, extend="both")

    for lev, col in zip(contour_levels, contour_colors):
        ax.contour(XX, YY, ZZ, levels=[lev], colors=[col], linewidths=1.0)

    ax.text(
        0.98,
        0.02,
        contour_text,
        transform=ax.transAxes,
        fontsize=8,
        ha="right",
        va="bottom",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.65),
    )

    ax.scatter(
        x[~selected_mask],
        y[~selected_mask],
        s=10,
        color="#d9d9d9",
        alpha=0.55,
        edgecolors="none",
        label="PDB references",
    )
    ax.scatter(
        x[selected_mask],
        y[selected_mask],
        s=90,
        marker="*",
        color="#f1c40f",
        edgecolors="black",
        linewidths=0.5,
        label="mdRNAcat",
        zorder=5,
    )

    ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    if show_legend:
        ax.legend(ncols=1, loc=legend_loc)
    padx = max(float(np.ptp(x)), 1e-6) * 0.035
    pady = max(float(np.ptp(y)), 1e-6) * 0.035
    ax.format(xticks=[], yticks=[], grid=False,
              xlim=(float(np.min(x)) - padx, float(np.max(x)) + padx),
              ylim=(float(np.min(y)) - pady, float(np.max(y)) + pady))

    return cf


In [ ]:
seq_xy = umap.UMAP(n_neighbors=min(35, len(X_seq)-1), min_dist=0.1,
                   metric="euclidean", random_state=SEED, n_jobs=1,
                   n_epochs=350).fit_transform(X_seq)
struct_xy = umap.UMAP(n_neighbors=min(48, len(X_struct)-1), min_dist=0.35,
                      metric="euclidean", random_state=SEED, n_jobs=1,
                      init="random", n_epochs=350).fit_transform(X_struct)
fig, axs = uplt.subplots([[1, 2, 3], [4, 4, 4]], figsize=(14, 6.5),
                         wratios=[1, 1.05, 1.05], hratios=[1, 0.45],
                         share=False, abc="a", hspace="3em", wspace="7em")
for curve, color, label in [(seq_curve, "#1f4e79", "Sequence"),
                             (struct_curve, "#097823", "Structural motifs")]:
    x = curve.threshold.to_numpy()
    y = curve.coverage.to_numpy()
    axs[0].fill_between(x, curve.ci_low.to_numpy(), curve.ci_high.to_numpy(), color=color, alpha=0.18, label=f"{label} 95% CI")
    axs[0].plot(x, y, color=color, lw=2, label=label)
    for t in [1.2, 1.5, 2.0]:
        value = float(curve.loc[np.isclose(curve.threshold, t), "coverage"].iloc[0])
        axs[0].scatter([t], [value], color=color, s=20)
        axs[0].annotate(f"{value:.2f}", (t, value), xytext=(-18 if label == "Sequence" else 2, 5 if label == "Sequence" else -15),
                        textcoords="offset points", color=color, fontsize=8)
axs[0].format(xlabel="Normalized local distance threshold", ylabel="Coverage fraction",
              xlim=(0.47, 3.03), ylim=(0, 1.03), grid=True)
axs[0].legend(ncols=1, loc="lower right", fontsize=8)
cs = plot_surface_panel(axs[1], *seq_xy.T, seq_ratio, seq_df.is_selected.to_numpy(),
                        "Sequence space", "viridis_r", [0.6, 0.8, 1, 1.2, 1.5, 2, 3],
                        [1.2, 1.5, 2], ["#f39c12", "#e74c3c", "#8e44ad"],
                        "Contours: 1.2, 1.5, 2.0")
levels = structural_levels_from_data(winsorize_finite(struct_ratio, 0.04, 0.92))
if len(levels) < 7:
    clipped = winsorize_finite(struct_ratio, 0.04, 0.92)
    finite = clipped[np.isfinite(clipped)]
    lo, hi = float(finite.min()), float(finite.max())
    levels = np.linspace(lo, max(hi, lo + 1e-6), 7).tolist()
contours = [levels[2], levels[4], levels[5]]
ct = plot_surface_panel(axs[2], *struct_xy.T, struct_ratio, selected_struct,
                        "Structural-motif space", "magma_r", levels, contours,
                        ["#00a6ca", "#f18f01", "#c73e1d"],
                        "Contours: " + ", ".join(f"{v:.2f}" for v in contours),
                        surf_k_override=56, mask_quantile_override=0.995,
                        support_radius_factor=3.0, clip_quantiles=(0.04, 0.92))
axs[1].colorbar(cs, loc="r", label="Sequence distance ratio", width="0.12in")
axs[2].colorbar(ct, loc="r", label="Structural distance ratio", width="0.12in")
axs[1].legend(loc="b", ncols=2, fontsize=8, frame=True)
x = np.arange(len(shifts))
axs[3].bar(x, shifts.to_numpy(), color=np.where(shifts.to_numpy() >= 0, "#2f7f5f", "#b03a2e"))
axs[3].axhline(0, color="black", lw=0.8)
axs[3].format(xticks=x, xticklabels=shifts.index.tolist(), xrotation=25,
              ylabel=r"$(\mu_{dataset}-\mu_{PDB})/\sigma_{PDB}$",
              xlabel="Structural-motif descriptor", grid=True, ylim=(-1, 2))
fig.savefig(RESULTS / "coverage.pdf", metadata={"Creator": "", "Author": "", "CreationDate": None, "ModDate": None})
fig.savefig(RESULTS / "coverage.png", dpi=250, metadata={"Software": ""})
packages = ["numpy", "pandas", "scipy", "scikit-learn", "umap-learn", "ultraplot", "requests"]
manifest = dict(config=CONFIG, selected_pdb_ids=sorted(selected_ids), features=features,
                sequence_svd_components=n_components, seed=SEED,
                packages={p: version(p) for p in packages},
                input_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(DATA.glob("*")) if p.is_file()},
                cache_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(CACHE.glob("*.txt"))})
(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2))
display(fig)
uplt.close(fig)  # Prevent a second automatic notebook display.
